In [1]:
#setup notebook
%load_ext autoreload
%autoreload 2
%matplotlib inline
#%matplotlib notebook
#%autosave 0
import base64
import pandas as pd
import numpy as np
import uuid
import os
import sys
import logging
logging.basicConfig(format='%(asctime)s %(levelname)s:%(message)s', level=logging.WARNING, datefmt='%I:%M:%S')

import sdata
from sdata import SUUID
from sdata.base import Base, sdata_factory
from sdata.sclass.dependency_graph import toposort, toposort_flatten

The input to the toposort function is a dict describing the
dependencies among the input nodes. Each key is a dependent node, the
corresponding value is a set containing the dependent nodes.

## Process order

The interpretation of the input data here is: If 2 depends on 11; 9
depends on 11, 8 and 10; 10 depends on 11 and 3 (and so on), then in what
order should we process the items such that all nodes are processed
before any of their dependencies?

In [4]:
seq = list(toposort({2: {11},
        9: {11, 8, 10},
        10: {11, 3},
        11: {7, 5},
        8: {7, 3},
        }))
seq

[{3, 5, 7}, {8, 11}, {2, 10}, {9}]

And the answer is: process 3, 5, and 7 (in any order); then process 8
and 11; then process 2 and 10; then process 9. Note that 3, 5, and 7
are returned first because they do not depend on anything. They are
then removed from consideration, and then 8 and 11 don't depend on
anything remaining. This process continues until all nodes are
returned, or a circular dependency is detected.

## Circular dependencies
A circular dependency will raise a CyclicDependencyError, which is
derived from ValueError. Here 1 depends on 2, and 2 depends on 1.

In [3]:
list(toposort({1: {2}, 
               2: {1},
              }))

CircularDependencyError: Circular dependencies exist among these items: {1: [2], 2: [1]}